# v345 Private Score 재현 실행

이 노트북은 동봉된 Python 학습·검증 진입점을 실행하는 래퍼입니다. 대회 제공 데이터 4개를 `code/data/`에 배치한 뒤 위에서 아래로 실행하세요. 구성요소별 환경과 전체 명령은 `01_ENVIRONMENT.md`와 `03_REPRODUCE.md`를 따릅니다.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / 'code', cwd.parent, cwd.parent / 'code']
CODE_ROOT = next((p for p in candidates if (p / 'train.py').is_file()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('code/train.py를 찾지 못했습니다. 제출 폴더 안에서 노트북을 실행하세요.')
PACKAGE_ROOT = CODE_ROOT.parent
DATA_DIR = CODE_ROOT / 'data'
print('Python:', sys.version)
print('PACKAGE_ROOT:', PACKAGE_ROOT)
print('DATA_DIR:', DATA_DIR)

In [ ]:
def run_python(*arguments: object, cwd: Path = CODE_ROOT) -> None:
    command = [sys.executable, *map(str, arguments)]
    print('RUN:', subprocess.list2cmdline(command))
    subprocess.run(command, cwd=cwd, check=True)

## 1. 공식 데이터·체크포인트·현재 환경 preflight

`train.csv`와 `trackman_history.csv`의 SHA-256, 동봉 체크포인트 해시 및 설치된 라이브러리 버전을 검사합니다.

In [ ]:
run_python('train.py', 'preflight', '--data-dir', DATA_DIR)

## 2. 코드 제출 구성 검사

PPT와 Phase 3 참가 여부는 별도 첨부하므로 여기서는 코드 범위만 검사합니다. Exact fresh-fit parity 미확인 사항은 필수 구조 검사와 분리해 진단으로 표시됩니다.

In [ ]:
run_python(CODE_ROOT / 'check_submission_contents.py', '--package-dir', PACKAGE_ROOT, '--code-only')

## 3. 선택: 전체 회귀 테스트

아래 값을 `True`로 바꾸면 전체 테스트를 실행합니다. 실행 기준 결과는 `verification/tests_summary.json`에 기록돼 있습니다.

In [ ]:
RUN_FULL_TESTS = False
if RUN_FULL_TESTS:
    run_python('-m', 'pytest', 'tests', '-q')
else:
    print('건너뜀: 실행하려면 RUN_FULL_TESTS = True로 변경하세요.')

## 4. 선택: 제출 v345 체크포인트 정확 재빌드

동봉된 v343 체크포인트와 공식 train에서 Beta 구성요소를 다시 적합하고, 결과 ZIP의 SHA-256이 원 제출 v345와 일치하는지 검사합니다. 이는 모든 상위 모델을 원데이터부터 한 번에 재학습하는 명령과는 구분됩니다. 출력 폴더는 존재하지 않아야 합니다.

In [ ]:
RUN_EXACT_CHECKPOINT_REBUILD = False
REBUILD_OUTPUT = PACKAGE_ROOT / 'notebook_outputs' / 'v345_exact_rebuild'
if RUN_EXACT_CHECKPOINT_REBUILD:
    run_python('train.py', 'rebuild', '--data-dir', DATA_DIR, '--output-dir', REBUILD_OUTPUT)
else:
    print('건너뜀: 실행하려면 RUN_EXACT_CHECKPOINT_REBUILD = True로 변경하세요.')

## 구성요소별 신규 학습

H1, strict, Futures, fallback XGBoost와 전체 계보의 세부 명령은 실행 환경이 서로 다르므로 이 노트북에서 일괄 실행하지 않습니다. `03_REPRODUCE.md`의 0~13절을 순서대로 따르세요. 특히 H1은 `code/h1/requirements_verified.txt`, fallback XGBoost는 XGBoost 3.2.0 CUDA 환경을 사용합니다.